In [17]:
import pandas as pd 
import json 
from tqdm.auto import tqdm
from openai import OpenAI

In [18]:


client = OpenAI()

In [19]:
df = pd.read_csv('../data/cleaned_data.csv')

In [20]:

documents = df.to_dict(orient='records')

In [21]:
print(documents[0])

{'id': 1, 'task': 'Write a 2-page project summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': 45, 'framework_name': 'Time Blocking', 'reasoning': 'Time Blocking helps allocate a clear writing window.', 'instructions': 'Block 45 minutes, outline key points, write summary, revise.', 'tags': 'work;writing;planning'}


## Evaluation

In [23]:
prompt_template = """
You emulate a user of our Productivity Advisor application.
Formulate 5 questions this user might ask based on a provided productivity task.
Make the questions specific to this task.
The questions should be complete, practical, and not too short.
Use as few exact words as possible from the record.

The record:

task: {task}
category: {category}
difficulty: {difficulty}
duration_estimate: {duration_estimate}
instructions: {instructions}
reasoning: {reasoning}
tags: {tags}

Provide the output in parsable JSON without using code blocks:

{{"questions": ["question1", "question2", ..., "question5"]}}

""".strip()

In [25]:
prompt = prompt_template.format(**documents[0])

print(prompt)

You emulate a user of our Productivity Advisor application.
Formulate 5 questions this user might ask based on a provided productivity task.
Make the questions specific to this task.
The questions should be complete, practical, and not too short.
Use as few exact words as possible from the record.

The record:

task: Write a 2-page project summary
category: work
difficulty: medium
duration_estimate: 45
instructions: Block 45 minutes, outline key points, write summary, revise.
reasoning: Time Blocking helps allocate a clear writing window.
tags: work;writing;planning

Provide the output in parsable JSON without using code blocks:

{"questions": ["question1", "question2", ..., "question5"]}


In [26]:
model='gpt-4o-mini'
def llm(prompt, model=model):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [31]:
questions = llm(prompt)
print(questions)

{"questions": ["What specific key points should I focus on while outlining for the project summary?", "How can I effectively utilize the 45-minute time block to maximize my writing productivity?", "What strategies can I employ to ensure that my summary is concise yet informative for my audience?", "Are there any recommended tools or techniques for revising and editing the summary once I have written the first draft?", "How can I minimize distractions during the 45 minutes I set aside for writing the project summary?"]}


In [32]:
import json

print(json.loads(questions))

{'questions': ['What specific key points should I focus on while outlining for the project summary?', 'How can I effectively utilize the 45-minute time block to maximize my writing productivity?', 'What strategies can I employ to ensure that my summary is concise yet informative for my audience?', 'Are there any recommended tools or techniques for revising and editing the summary once I have written the first draft?', 'How can I minimize distractions during the 45 minutes I set aside for writing the project summary?']}


In [33]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

In [34]:

results = {}

In [35]:
for doc in tqdm(documents): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions_raw = generate_questions(doc)
    questions = json.loads(questions_raw)
    results[doc_id] = questions['questions']

  0%|          | 0/250 [00:00<?, ?it/s]

In [36]:
final_results = []

for doc_id, questions in results.items():
    for q in questions:
        final_results.append((doc_id, q))

In [37]:
final_results[0]

(1, 'What key points should I focus on when outlining the project summary?')

In [38]:
df_results = pd.DataFrame(final_results, columns=['id', 'question'])

In [39]:
df_results.to_csv('../data/ground-truth-retrieval.csv', index=False)

In [40]:
!head ../data/ground-truth-retrieval.csv

id,question
1,What key points should I focus on when outlining the project summary?
1,How can I effectively block out 45 minutes in my schedule for this task?
1,What strategies can I use to ensure the summary stays concise while covering all essential information?
1,What should I include in the revision process to improve the overall quality of the summary?
1,Are there any specific techniques for time blocking that can enhance my writing productivity during this task?
2,What specific categories should I use to sort items in my pantry when organizing the shelves?
2,Are there any recommended cleaning supplies or methods for effectively cleaning shelf surfaces before restocking?
2,"How do I handle items that I haven't used in a while—should I keep, donate, or discard them?"
2,What strategies can I implement to maintain the organization of my pantry after I reorganize it?
